# 4-Stage Curriculum Fine-tuning (Unsloth + LoRA) on Colab

This notebook runs a 4-stage fine-tuning pipeline for DeepSeek-R1-Distill-Llama-8B using Unsloth and LoRA:

- Stage 1: Foundation
- Stage 2: Algorithms
- Stage 3: Debugging
- Stage 4: Advanced Reasoning

Each stage saves its output in a separate folder, and the next stage continues from the previous stage’s model.


In [ ]:
# If GPU is not enabled, go to Runtime → Change runtime type → pick GPU.
python - << 'PY'
import torch
print('CUDA available:', torch.cuda.is_available())
PY

pip -q install --upgrade pip
pip -q install unsloth transformers datasets accelerate peft bitsandbytes trl


In [ ]:
pip -q install google-generativeai openai


In [ ]:
# Optional: Mount Google Drive to save models
USE_DRIVE = False  # Set True to save to Drive
DRIVE_DIR = "/content/drive/MyDrive/colab_models/curriculum_finetune"

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

import os
os.makedirs(DRIVE_DIR, exist_ok=True) if USE_DRIVE else None


In [ ]:
# T4-friendly configuration overrides
CONFIG = {
    "hardware_profile": "t4",
    "gradient_accumulation_steps": 16,   # Increase to simulate larger batch on small GPU
    "save_total_limit": 2,               # Limit checkpoints to save space
    "stages": {
        "foundation": {"max_seq": 2048, "batch_size": 2, "epochs": 1},
        "algorithms": {"max_seq": 2048, "batch_size": 1, "epochs": 1},
        "debugging":  {"max_seq": 2048, "batch_size": 1, "epochs": 1},
        "advanced":   {"max_seq": 2048, "batch_size": 1, "epochs": 1},
    },
}

def apply_overrides(stage_cfg, overrides):
    out = dict(stage_cfg)
    for k, v in (overrides or {}).items():
        out[k] = v
    return out


In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset, load_from_disk
from trl import SFTTrainer
from transformers import TrainingArguments
import os
import torch

class CurriculumFineTuner:
    def __init__(self, base_model="unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"):
        self.base_model = base_model
        self.stages = [
            {
                "name": "foundation",
                "dataset": "iamtarun/python_code_instructions_18k_alpaca",
                "max_seq": 2048,
                "lora_r": 32,
                "epochs": 2,
                "lr": 3e-4,
                "batch_size": 4,
            },
            {
                "name": "algorithms",
                "dataset": "codeparrot/apps",
                "max_seq": 3072,
                "lora_r": 48,
                "epochs": 3,
                "lr": 2e-4,
                "batch_size": 2,
                "filter": lambda x: x.get("difficulty") in ["introductory", "interview"],
            },
            {
                "name": "debugging",
                "dataset": "m-a-p/Code-Feedback",
                "max_seq": 4096,
                "lora_r": 64,
                "epochs": 2,
                "lr": 1e-4,
                "batch_size": 2,
            },
            {
                "name": "advanced",
                "dataset": "ise-uiuc/Magicoder-Evol-Instruct-110K",
                "max_seq": 4096,
                "lora_r": 64,
                "epochs": 3,
                "lr": 5e-5,
                "batch_size": 2,
            },
        ]

    def format_stage1(self, example):
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a Python programming instructor who writes clear and simple code.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
{example['instruction']}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
{example['output']}<|eot_id|>"""
        return {"text": prompt}

    def format_stage2(self, example):
        solution = example['solutions'][0] if example.get('solutions') else "# Solution not available"
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an algorithms expert who solves problems with step-by-step reasoning.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Problem:
{example['question']}

Solve this problem.<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
Let's analyze step by step:

```python
{solution}
```<|eot_id|>"""
        return {"text": prompt}

    def format_stage3(self, example):
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a code review and debugging expert.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Review this code:

```python
{example['code']}
```

Issue: {example['query']}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
**Issues:**
{example['feedback']}

**Corrected code:**

```python
{example['corrected_code']}
```<|eot_id|>"""
        return {"text": prompt}

    def format_stage4(self, example):
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert programmer with deep reasoning ability.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
{example['instruction']}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
{example['response']}<|eot_id|>"""
        return {"text": prompt}

    def train_stage(self, stage_idx, prev_model=None):
        stage = self.stages[stage_idx]
        print(f"\n{'='*60}")
        print(f"🚀 Stage {stage_idx + 1}: {stage['name'].upper()}")
        print(f"{'='*60}\n")

        model_name = prev_model if prev_model else self.base_model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=stage['max_seq'],
            dtype=torch.float16,
            load_in_4bit=True,
            device_map="auto",
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r=stage['lora_r'],
            lora_alpha=stage['lora_r'] * 2,
            lora_dropout=0.05,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
            use_gradient_checkpointing="unsloth",
        )

        # Dataset paths
        processed_dir = f"./stage{stage_idx + 1}-{stage['name']}-dataset"
        print(f"📥 Loading dataset: {stage['dataset']}")

        # Load processed dataset if exists; otherwise load, format, and save
        if os.path.exists(processed_dir):
            print(f"🔁 Found processed dataset. Loading from {processed_dir}")
            dataset = load_from_disk(processed_dir)
        else:
            dataset = load_dataset(stage['dataset'], split="train")
            if 'filter' in stage:
                dataset = dataset.filter(stage['filter'])
            format_func = getattr(self, f"format_stage{stage_idx + 1}")
            dataset = dataset.map(format_func)
            print(f"🗂️ Saving processed dataset to {processed_dir}")
            dataset.save_to_disk(processed_dir)

        output_dir = f"./stage{stage_idx + 1}-{stage['name']}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=stage['batch_size'],
            gradient_accumulation_steps=8,
            warmup_steps=50 * (stage_idx + 1),
            num_train_epochs=stage['epochs'],
            learning_rate=stage['lr'],
            fp16=True,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            save_strategy="epoch",
            save_total_limit=2,
            report_to="none",
        )

        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            args=training_args,
            max_seq_length=stage['max_seq'],
            dataset_text_field="text",
        )

        print(f"🎯 Starting training...")
        trainer.train(resume_from_checkpoint=True)

        print(f"💾 Saving model to {output_dir}")
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

        if 'filter' in stage:
            print(f"ℹ️ Number of samples after filter: {len(dataset)}")

        print(f"✅ Stage {stage_idx + 1} completed!")
        return output_dir

    def run_full_curriculum(self):
        print("\n" + "🎓" * 30)
        print("Starting 4-stage fine-tuning")
        print("🎓" * 30 + "\n")

        prev_model = None
        for i in range(len(self.stages)):
            prev_model = self.train_stage(i, prev_model)

        print("\n" + "🎉" * 30)
        print("All 4 stages completed successfully!")
        print(f"Final model: {prev_model}")
        print("🎉" * 30 + "\n")
        return prev_model


In [ ]:
# Distillation helpers: Teacher client (Gemini + OpenAI-compatible) and subclassed trainer
import os, time
from typing import Optional

# Env vars expected (set in Colab secrets or manually):
# - GOOGLE_API_KEY           (for Gemini models)
# - OPENAI_API_KEY           (for OpenAI-compatible endpoints like OpenRouter/Together)
# - OPENAI_BASE_URL          (e.g., "https://openrouter.ai/api/v1" or your Ollama endpoint)

import google.generativeai as genai
from openai import OpenAI

TEACHER_PLANNING_MODELS = [
    "gemini-2.5-pro",
    "gemini-2.5-pro-preview-03-25",
    "gemini-2.5-pro-preview-05-06",
    "gemini-2.5-pro-preview-06-05",
    "gemini-pro-latest",
    "deepseek-v3.1:671b-cloud",
    "kimi-k2:1t-cloud",
    "qwen3-coder:480b-cloud",
    "gpt-oss:120b-cloud",
    "glm-4.6:cloud",
    "minimax-m2:cloud",
]

def is_ioi_example(ex):
    parts = [
        str(ex.get("question", "")),
        str(ex.get("title", "")),
        str(ex.get("source", "")),
        str(ex.get("tags", "")),
        str(ex.get("url", "")),
    ]
    return "ioi" in " ".join(parts).lower()

class TeacherClient:
    def __init__(self):
        self.google_api_key = os.environ.get("GOOGLE_API_KEY")
        self.openai_api_key = os.environ.get("OPENAI_API_KEY")
        self.openai_base_url = os.environ.get("OPENAI_BASE_URL")
        # Configure Gemini client if key present
        if self.google_api_key:
            genai.configure(api_key=self.google_api_key)
        # Configure OpenAI-compatible client if provided
        self.openai_client = None
        if self.openai_api_key and self.openai_base_url:
            self.openai_client = OpenAI(api_key=self.openai_api_key, base_url=self.openai_base_url)

    def _is_gemini(self, model: str) -> bool:
        return model.startswith("gemini")

    def generate(self, model: str, prompt: str, max_tokens: int = 1024, temperature: float = 0.7) -> str:
        if self._is_gemini(model):
            if not self.google_api_key:
                raise RuntimeError("GOOGLE_API_KEY is not set for Gemini models.")
            m = genai.GenerativeModel(model)
            resp = m.generate_content(prompt, generation_config={"max_output_tokens": max_tokens, "temperature": temperature})
            return getattr(resp, "text", "").strip()
        else:
            if not self.openai_client:
                raise RuntimeError("OPENAI_API_KEY and OPENAI_BASE_URL must be set for non-Gemini teacher models.")
            comp = self.openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful, expert coding assistant."},
                    {"role": "user", "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return comp.choices[0].message.content.strip()

from datasets import Dataset

class DistillCurriculumFineTuner(CurriculumFineTuner):
    def __init__(self, *args, teacher_model: str, distill_max_samples: Optional[int] = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = TeacherClient()
        self.teacher_model = teacher_model
        self.distill_max_samples = distill_max_samples

    # Build stage-specific teacher prompts (user-only)
    def _teacher_prompt(self, idx: int, ex: dict) -> str:
        if idx == 0:
            return f"""You are a Python programming instructor who writes clear and simple code.
Instruction:
{ex.get('instruction','')}
Respond with a clear, concise Python-focused answer."""
        if idx == 1:
            # Prefer IOI algorithm view if present
            if isinstance(ex.get('algorithm_view'), dict):
                av = ex['algorithm_view']
                parts = []
                title = av.get('title') or ''
                one_line = av.get('one_line') or ''
                given = "\n".join(av.get('given', [])) if isinstance(av.get('given'), list) else str(av.get('given',''))
                goal = av.get('goal') or ''
                mechanism = av.get('mechanism') or ''
                constraints = av.get('constraints', {})
                critical = constraints.get('critical', []) if isinstance(constraints, dict) else []
                parts.append(f"Title: {title}")
                if one_line:
                    parts.append(f"Summary: {one_line}")
                if given:
                    parts.append(f"Given:\n{given}")
                if goal:
                    parts.append(f"Goal: {goal}")
                if mechanism:
                    parts.append(f"Mechanism: {mechanism}")
                if critical:
                    parts.append("Key constraints:\n- " + "\n- ".join(critical))
                body = "\n\n".join([p for p in parts if p])
                return f"""You are an algorithms expert. Solve the following IOI-style problem step-by-step with clear reasoning and provide a correct Python solution.
{body}
Return only your reasoning and the final Python solution."""
            q = ex.get('question', '')
            return f"""You are an algorithms expert. Solve the problem step-by-step with reasoning and provide a correct Python solution.
Problem:
{q}
Return only your solution and reasoning."""
        if idx == 2:
            return f"""You are a code review and debugging expert.
Analyze the following code and the issue; list issues and provide corrected code.
Code:
{ex.get('code','')}
Issue:
{ex.get('query','')}"""
        return f"""You are an expert programmer with deep reasoning ability.
Instruction:
{ex.get('instruction','')}
Provide the best possible answer."""

    # Build final training text with teacher answer
    def _build_text(self, idx: int, ex: dict, teacher_answer: str) -> str:
        if idx == 0:
            return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Python programming instructor who writes clear and simple code.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{ex.get('instruction','')}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{teacher_answer}<|eot_id|>"""
        if idx == 1:
            # Build user block from IOI if present; else fallback to 'question'
            if isinstance(ex.get('algorithm_view'), dict):
                av = ex['algorithm_view']
                parts = []
                title = av.get('title') or ''
                one_line = av.get('one_line') or ''
                given = "\n".join(av.get('given', [])) if isinstance(av.get('given'), list) else str(av.get('given',''))
                goal = av.get('goal') or ''
                mechanism = av.get('mechanism') or ''
                constraints = av.get('constraints', {})
                critical = constraints.get('critical', []) if isinstance(constraints, dict) else []
                parts.append(f"Title: {title}")
                if one_line:
                    parts.append(f"Summary: {one_line}")
                if given:
                    parts.append(f"Given:\n{given}")
                if goal:
                    parts.append(f"Goal: {goal}")
                if mechanism:
                    parts.append(f"Mechanism: {mechanism}")
                if critical:
                    parts.append("Key constraints:\n- " + "\n- ".join(critical))
                user_block = "\n\n".join([p for p in parts if p])
            else:
                user_block = f"Problem:\n{ex.get('question','')}\nSolve this problem."
            return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an algorithms expert who solves problems with step-by-step reasoning.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_block}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{teacher_answer}<|eot_id|>"""
        if idx == 2:
            return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a code review and debugging expert.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Review this code:

```python
{ex.get('code','')}
```

Issue: {ex.get('query','')}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{teacher_answer}<|eot_id|>"""
        return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert programmer with deep reasoning ability.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{ex.get('instruction','')}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{teacher_answer}<|eot_id|>"""

    def train_stage(self, idx, prev_model=None):
        st = self.stages[idx]
        print("\n" + "="*60)
        print(f"🚀 Stage {idx+1} (distill): {st['name'].upper()} :: Teacher = {self.teacher_model}")
        print("="*60 + "\n")

        model_name = prev_model or self.base_model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=st["max_seq"],
            dtype=torch.float16,
            load_in_4bit=True,
            device_map="auto",
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=st["lora_r"],
            lora_alpha=st["lora_r"] * 2,
            lora_dropout=0.05,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            use_gradient_checkpointing="unsloth",
        )

        proc_dir = f"./stage{idx+1}-{st['name']}-dataset"
        out_dir  = f"./stage{idx+1}-{st['name']}"

        # Load or prepare base dataset
        if st["name"] == "algorithms" and st.get("use_local_ioi"):
            path = st.get("ioi_json_path") or "ioi_multi_view.json"
            print(f"📥 Loading local IOI dataset from {path}")
            base_ds = load_dataset("json", data_files=path, split="train")
        else:
            print(f"📥 Dataset hub: {st['dataset']}")
            if os.path.exists(proc_dir):
                print(f"🔁 Using cached processed dataset: {proc_dir}")
                base_ds = load_from_disk(proc_dir)
            else:
                base_ds = load_dataset(st["dataset"], split="train")
                if "filter" in st:
                    base_ds = base_ds.filter(st["filter"])
                # Save raw/base for reuse
                base_ds.save_to_disk(proc_dir)
                print(f"🗂️ Saved raw/base dataset to {proc_dir}")

        # Optional IOI-only filtering (when using hub datasets)
        if st.get("ioi_only") and not st.get("use_local_ioi"):
            base_ds = base_ds.filter(is_ioi_example)

        # Optionally limit samples for cost/time
        if self.distill_max_samples is not None:
            base_ds = base_ds.select(range(min(self.distill_max_samples, len(base_ds))))

        # Distill: query teacher and build final 'text'
        print("🧠 Distilling from teacher... This may take a while.")
        texts = []
        for ex in base_ds:
            prompt = self._teacher_prompt(idx, ex)
            try:
                ans = self.teacher.generate(self.teacher_model, prompt, max_tokens=1024, temperature=0.7)
            except Exception as e:
                ans = ""
                print("Teacher error:", e)
            texts.append(self._build_text(idx, ex, ans))
            time.sleep(0.05)  # gentle pacing
        distilled = Dataset.from_dict({"text": texts})
        dist_dir = f"./stage{idx+1}-{st['name']}-distilled"
        distilled.save_to_disk(dist_dir)
        print(f"✅ Distilled dataset saved to {dist_dir}")

        args = TrainingArguments(
            output_dir=out_dir,
            per_device_train_batch_size=st["batch_size"],
            gradient_accumulation_steps=self.grad_acc,
            num_train_epochs=st["epochs"],
            learning_rate=st["lr"],
            warmup_steps=50*(idx+1),
            fp16=True,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            save_strategy="epoch",
            save_total_limit=CONFIG.get("save_total_limit", 2),
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            report_to="none",
        )
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=distilled,
            args=args,
            max_seq_length=st["max_seq"],
            dataset_text_field="text",
        )
        eff_bsz = st["batch_size"] * self.grad_acc
        print(f"🎯 Start training on distilled data (effective batch size = {eff_bsz})")
        trainer.train(resume_from_checkpoint=True)

        print(f"💾 Saving model → {out_dir}")
        model.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        print(f"✅ Stage {idx+1} done.")
        return out_dir

        dist_dir = f"./stage{idx+1}-{st['name']}-distilled"
        print(f"📥 Dataset hub: {st['dataset']}")

        # Load or prepare base dataset
        if os.path.exists(proc_dir):
            print(f"🔁 Using cached processed dataset: {proc_dir}")
            base_ds = load_from_disk(proc_dir)
        else:
            base_ds = load_dataset(st["dataset"], split="train")
            if "filter" in st:
                base_ds = base_ds.filter(st["filter"])
            # No formatting yet; we will build prompts for teacher
            base_ds.save_to_disk(proc_dir)
            print(f"🗂️ Saved raw/base dataset to {proc_dir}")

        # Optionally limit samples for cost/time
        if self.distill_max_samples is not None:
            base_ds = base_ds.select(range(min(self.distill_max_samples, len(base_ds))))

        # Distill: query teacher and build final 'text'
        print("🧠 Distilling from teacher... This may take a while.")
        texts = []
        for ex in base_ds:
            prompt = self._teacher_prompt(idx, ex)
            try:
                ans = self.teacher.generate(self.teacher_model, prompt, max_tokens=1024, temperature=0.7)
            except Exception as e:
                ans = ""
                print("Teacher error:", e)
            texts.append(self._build_text(idx, ex, ans))
            time.sleep(0.05)  # gentle pacing
        distilled = Dataset.from_dict({"text": texts})
        distilled.save_to_disk(dist_dir)
        print(f"✅ Distilled dataset saved to {dist_dir}")

        args = TrainingArguments(
            output_dir=out_dir,
            per_device_train_batch_size=st["batch_size"],
            gradient_accumulation_steps=self.grad_acc,
            num_train_epochs=st["epochs"],
            learning_rate=st["lr"],
            warmup_steps=50*(idx+1),
            fp16=True,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            save_strategy="epoch",
            save_total_limit=CONFIG.get("save_total_limit", 2),
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            report_to="none",
        )
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=distilled,
            args=args,
            max_seq_length=st["max_seq"],
            dataset_text_field="text",
        )
        eff_bsz = st["batch_size"] * self.grad_acc
        print(f"🎯 Start training on distilled data (effective batch size = {eff_bsz})")
        trainer.train(resume_from_checkpoint=True)

        print(f"💾 Saving model → {out_dir}")
        model.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        print(f"✅ Stage {idx+1} done.")
        return out_dir


In [ ]:
# Optional: mount Drive for saving (set True to use)
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/colab_models/curriculum_finetune"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

# T4-friendly config you can tweak
CONFIG = {
    "gradient_accumulation_steps": 16,  # effective batch size = per_device * this
    "save_total_limit": 2,              # limit checkpoints
    "stages": {
        "foundation": {"max_seq": 2048, "batch_size": 2, "epochs": 1},
        # Set ioi_only=True to select IOI-style problems in algorithms stage
        # Set use_local_ioi=True to load from a local JSON dataset (ioi_multi_view.json)
        "algorithms": {
            "max_seq": 2048,
            "batch_size": 1,
            "epochs": 1,
            "ioi_only": False,
            "use_local_ioi": False,
            "ioi_json_path": "ioi_multi_view.json",
        },
        "debugging":  {"max_seq": 2048, "batch_size": 1, "epochs": 1},
        "advanced":   {"max_seq": 2048, "batch_size": 1, "epochs": 1},
    },
}

def apply_overrides(stage_cfg, overrides):
    out = dict(stage_cfg)
    if overrides:
        out.update(overrides)
    return out

def is_ioi_example(ex):
    # Heuristic: check typical text fields for "ioi" marker
    parts = [
        str(ex.get("question", "")),
        str(ex.get("title", "")),
        str(ex.get("source", "")),
        str(ex.get("tags", "")),
        str(ex.get("url", "")),
    ]
    hay = " ".join(parts).lower()
    return "ioi" in hay

from unsloth import FastLanguageModel
from datasets import load_dataset, load_from_disk
from trl import SFTTrainer
from transformers import TrainingArguments
import os, torch

class CurriculumFineTuner:
    def __init__(self, base_model="unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"):
        self.base_model = base_model
        # Base stage configs (overridden by CONFIG)
        self.stages = [
            {"name": "foundation", "dataset": "iamtarun/python_code_instructions_18k_alpaca", "max_seq": 2048, "lora_r": 32, "epochs": 2, "lr": 3e-4, "batch_size": 4},
            {"name": "algorithms", "dataset": "codeparrot/apps", "max_seq": 3072, "lora_r": 48, "epochs": 3, "lr": 2e-4, "batch_size": 2, "filter": lambda x: x.get("difficulty") in ["introductory", "interview"]},
            {"name": "debugging",  "dataset": "m-a-p/Code-Feedback", "max_seq": 4096, "lora_r": 64, "epochs": 2, "lr": 1e-4, "batch_size": 2},
            {"name": "advanced",   "dataset": "ise-uiuc/Magicoder-Evol-Instruct-110K", "max_seq": 4096, "lora_r": 64, "epochs": 3, "lr": 5e-5, "batch_size": 2},
        ]
        # Apply CONFIG
        self.grad_acc = int(CONFIG.get("gradient_accumulation_steps", 16))
        self.save_total_limit = int(CONFIG.get("save_total_limit", 2))
        stage_overrides = CONFIG.get("stages", {})
        self.stages = [apply_overrides(s, stage_overrides.get(s["name"])) for s in self.stages]

    def _format1(self, ex):
        return {"text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Python programming instructor who writes clear and simple code.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{ex['instruction']}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{ex['output']}<|eot_id|>"""}

    def _format2(self, ex):
        sol = ex["solutions"][0] if ex.get("solutions") else "# Solution not available"
        return {"text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an algorithms expert who solves problems with step-by-step reasoning.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Problem:
{ex['question']}
Solve this problem.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Let's analyze step by step:
```python
{sol}
```<|eot_id|>"""}

    def _format3(self, ex):
        return {"text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a code review and debugging expert.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Review this code:
```python
{ex['code']}
```
Issue: {ex['query']}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
**Issues:**
{ex['feedback']}
**Corrected code:**
```python
{ex['corrected_code']}
```<|eot_id|>"""}

    def _format4(self, ex):
        return {"text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert programmer with deep reasoning ability.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{ex['instruction']}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{ex['response']}<|eot_id|>"""}

    def train_stage(self, idx, prev_model=None):
        st = self.stages[idx]
        print("\n" + "="*60)
        print(f"🚀 Stage {idx+1}: {st['name'].upper()}")
        print("="*60 + "\n")

        model_name = prev_model or self.base_model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=st["max_seq"],
            dtype=torch.float16,
            load_in_4bit=True,
            device_map="auto",
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=st["lora_r"],
            lora_alpha=st["lora_r"] * 2,
            lora_dropout=0.05,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            use_gradient_checkpointing="unsloth",
        )

        proc_dir = f"./stage{idx+1}-{st['name']}-dataset"
        out_dir  = f"./stage{idx+1}-{st['name']}"
        print(f"📥 Dataset hub: {st['dataset']}")
        if os.path.exists(proc_dir):
            print(f"🔁 Using cached processed dataset: {proc_dir}")
            dataset = load_from_disk(proc_dir)
        else:
            dataset = load_dataset(st["dataset"], split="train")
            if "filter" in st:
                dataset = dataset.filter(st["filter"])
            # Optional IOI-only filtering for algorithms stage
            if st.get("ioi_only"):
                dataset = dataset.filter(is_ioi_example)
            fmt = [self._format1, self._format2, self._format3, self._format4][idx]
            dataset = dataset.map(fmt)
            print(f"🗂️ Saving processed dataset to {proc_dir}")
            dataset.save_to_disk(proc_dir)

        args = TrainingArguments(
            output_dir=out_dir,
            per_device_train_batch_size=st["batch_size"],
            gradient_accumulation_steps=self.grad_acc,
            num_train_epochs=st["epochs"],
            learning_rate=st["lr"],
            warmup_steps=50*(idx+1),
            fp16=True,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            save_strategy="epoch",
            save_total_limit=CONFIG.get("save_total_limit", 2),
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            report_to="none",
        )
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=dataset,
            args=args,
            max_seq_length=st["max_seq"],
            dataset_text_field="text",
        )
        eff_bsz = st["batch_size"] * self.grad_acc
        print(f"🎯 Start training (effective batch size = {eff_bsz})")
        trainer.train(resume_from_checkpoint=True)

        print(f"💾 Saving model → {out_dir}")
        model.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        print(f"✅ Stage {idx+1} done.")
        return out_dir

    def run(self):
        prev = None
        for i in range(len(self.stages)):
            prev = self.train_stage(i, prev)
        print("\n🎉 All stages completed. Final model:", prev)
        return prev


In [ ]:
# Execution plan:
# 1) Run 4-stage standard SFT
# 2) Then run knowledge distillation starting from the SFT result

RUN_SFT = True
RUN_DISTILL = True

TEACHER_MODEL = "gemini-2.5-pro"  # pick from TEACHER_PLANNING_MODELS
DISTILL_MAX_SAMPLES = 200          # limit per stage to control cost/time (None for all)

# Set env vars here or via Colab Secrets before running the next cells
# os.environ["GOOGLE_API_KEY"] = "..."     # for Gemini
# os.environ["OPENAI_API_KEY"] = "..."     # for OpenAI-compatible (OpenRouter/Together/Ollama proxy)
# os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"  # or your endpoint

sft_final_dir = None
final_model_dir = None


## 1) Run 4-stage Standard SFT


In [ ]:
if RUN_SFT:
    sft_runner = CurriculumFineTuner()
    sft_final_dir = sft_runner.run()
else:
    print("Skipping SFT phase as requested.")


## 2) Run Knowledge Distillation (uses SFT output if available)


In [ ]:
if RUN_DISTILL:
    base_for_distill = sft_final_dir if sft_final_dir else None
    if base_for_distill:
        print("Using SFT output as base for distillation:", base_for_distill)
        distill_runner = DistillCurriculumFineTuner(base_model=base_for_distill, teacher_model=TEACHER_MODEL, distill_max_samples=DISTILL_MAX_SAMPLES)
    else:
        distill_runner = DistillCurriculumFineTuner(teacher_model=TEACHER_MODEL, distill_max_samples=DISTILL_MAX_SAMPLES)
    final_model_dir = distill_runner.run()
else:
    final_model_dir = sft_final_dir


## 3) (Optional) Copy Final Model to Google Drive


In [ ]:
if USE_DRIVE and final_model_dir:
    import shutil, os
    tgt = os.path.join(DRIVE_DIR, os.path.basename(final_model_dir))
    if os.path.exists(tgt):
        shutil.rmtree(tgt)
    shutil.copytree(final_model_dir, tgt)
    print("📦 Copied to Drive:", tgt)
else:
    print("Drive copy skipped or no final model yet.")


## 4) Quick Test on Final Model


In [ ]:
if final_model_dir:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=final_model_dir,
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Write a function that finds the longest common subsequence of two strings.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    print(tokenizer.decode(outputs[0], skip_special_tokens=False))
else:
    print("No final model produced.")


In [ ]:
# Quick test of the final model (generation)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=final_model_dir,
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

test_prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Write a function that finds the longest common subsequence of two strings.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7)
result = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(result)
